<a href="https://colab.research.google.com/github/Noadori34/Web-analytics-project/blob/main/Final_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import userdata

API_KEY = userdata.get('API_KEY')  # <-- Paste your key inside the quotes
BASE_URL = "https://himalayas.app/jobs/api/search"

HEADERS = {
    "User-Agent": "UniversityResearchProject/1.0",
    "Authorization": f"Bearer {API_KEY}",
}

SEARCH_QUERIES = [
    "data scientist",
    "machine learning",
    "data engineer",
    "data analyst",
    "artificial intelligence",
    "applied scientist",
]

jobs_by_guid = {}

for query in SEARCH_QUERIES:
    print(f"Searching for: '{query}'...")
    page = 1

    while True:
        params = {
            "q": query,
            "country": "US",       # Remove or change if you want global listings
            "sort": "recent",
            "page": page,
        }

        resp = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=15)
        if resp.status_code == 429:
            print("Rate limited — pausing 10 seconds...")
            time.sleep(10)
            continue
        resp.raise_for_status()

        jobs = resp.json().get("jobs", [])
        if not jobs:
            break  # No more pages for this query

        for job in jobs:
            guid = job.get("guid")
            if guid and guid not in jobs_by_guid:
                # Convert HTML description to clean plain text
                raw_html = job.get("description", "")
                clean_text = BeautifulSoup(raw_html, "html.parser").get_text(separator=" ", strip=True)
                job["clean_description"] = " ".join(clean_text.split())
                jobs_by_guid[guid] = job

        page += 1
        time.sleep(1)  # Polite 1-second delay between pages

df = pd.DataFrame(jobs_by_guid.values())

# Inspect the unique categories returned so you can refine further if needed
print("Top parent categories:", df["parentCategories"].explode().value_counts().head(10))

# Filter to keep only jobs with explicit salary figures
df_with_salary = df[df["minSalary"].notna() | df["maxSalary"].notna()].copy()

# Save to CSV
cols = [
    "guid", "title", "companyName", "seniority", "employmentType",
    "categories", "parentCategories", "minSalary", "maxSalary",
    "salaryPeriod", "currency", "clean_description"
]
df_with_salary[cols].to_csv("himalayas_data_science_jobs.csv", index=False)
print(f"Saved {len(df_with_salary)} unique Data Science jobs with salary data.")

Searching for: 'data scientist'...
Searching for: 'machine learning'...
Searching for: 'data engineer'...
Searching for: 'data analyst'...
Searching for: 'artificial intelligence'...
Searching for: 'applied scientist'...
Top parent categories: parentCategories
Data Science         3045
Developer            2871
Sales                 539
Operations            487
Finance               372
Product               291
Marketing             289
Hardware Engineer     221
Customer Service      190
Healthcare            147
Name: count, dtype: int64
Saved 4538 unique Data Science jobs with salary data.


In [16]:
df_with_salary.iloc[0]

,0
title,Principal Data Analyst
excerpt,Our mission at HubSpot is to help millions of ...
companyName,HubSpot
companySlug,hubspot
companyLogo,https://cdn-images.himalayas.app/pgeb5qfnlbpwv...
employmentType,Full Time
minSalary,152200.0
maxSalary,243500.0
salaryPeriod,annual
seniority,[Senior]


In [17]:
import re

SKILL_PATTERNS = {
    # Programming Languages
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "R": r"\b[Rr]\b",  # Matches standalone 'R' or 'r'
    "C++": r"\bc\+\+|\bcpp\b",
    "C#": r"\bc\#|\bc-sharp\b",
    "Java": r"\bjava\b",
    "JavaScript": r"\bjavascript\b|\bjs\b",
    "TypeScript": r"\btypescript\b|\bts\b",
    "Go": r"\bgolang\b|\bgo\b",
    "Rust": r"\brust\b",
    "Scala": r"\bscala\b",
    "Bash/Shell": r"\bbash\b|\bshell\b",

    # AI, ML & Data Science
    "PyTorch": r"\bpytarch\b|\bpytorch\b",
    "TensorFlow": r"\btensorflow\b|\btf\b",
    "Keras": r"\bkeras\b",
    "Scikit-Learn": r"\bscikit-learn\b|\bsklearn\b",
    "Pandas": r"\bpandas\b",
    "NumPy": r"\bnumpy\b",
    "Apache Spark": r"\bspark\b|\bpyspark\b",
    "Hadoop": r"\bhadoop\b",
    "Airflow": r"\bairflow\b",
    "Databricks": r"\bdatabricks\b",
    "LangChain": r"\blangchain\b",
    "Hugging Face": r"\bhugging\s*face\b",
    "LLM / GenAI": r"\bllm\b|\bllms\b|\bgenai\b|\bgenerative ai\b|\btransformers\b",
    "Computer Vision": r"\bcomputer vision\b|\bopencv\b",
    "NLP": r"\bnlp\b|\bnatural language processing\b",

    # Cloud & DevOps
    "AWS": r"\baws\b|\bamazon web services\b",
    "Azure": r"\bazure\b",
    "GCP": r"\bgcp\b|\bgoogle cloud\b",
    "Docker": r"\bdocker\b",
    "Kubernetes": r"\bkubernetes\b|\bk8s\b",
    "Terraform": r"\bterraform\b",
    "Linux": r"\blinux\b|\bunix\b",
    "Git": r"\bgit\b|\bgithub\b|\bgitlab\b",
    "CI/CD": r"\bci\/cd\b|\bcontinuous integration\b",

    # Databases & Storage
    "PostgreSQL": r"\bpostgres\b|\bpostgresql\b",
    "MySQL": r"\bmysql\b",
    "MongoDB": r"\bmongodb\b|\bmongo\b",
    "Redis": r"\bredis\b",
    "Snowflake": r"\bsnowflake\b",
    "BigQuery": r"\bbigquery\b",
    "Redshift": r"\bredshift\b",

    # BI & Visualization
    "Tableau": r"\btableau\b",
    "Power BI": r"\bpower bi\b|\bpowerbi\b",
    "Looker": r"\blooker\b"
}

def extract_skills(description):
    """Scans job description text and returns a list of matched skills."""
    if not isinstance(description, str) or not description:
        return []

    found_skills = []
    text_lower = description.lower()

    for skill_name, pattern in SKILL_PATTERNS.items():
        if re.search(pattern, text_lower, re.IGNORECASE):
            found_skills.append(skill_name)

    return found_skills

# 2. Extract skills as a list column
df_with_salary['skills_list'] = df_with_salary['clean_description'].apply(extract_skills)

# 3. Create a comma-separated string column for easy display/CSV export
df_with_salary['skills'] = df_with_salary['skills_list'].apply(lambda x: ", ".join(x) if x else "None Specified")

# 4. View a sample of the results
df_with_salary

,title,excerpt,companyName,companySlug,companyLogo,employmentType,minSalary,maxSalary,salaryPeriod,seniority,...,categories,parentCategories,description,pubDate,expiryDate,applicationLink,guid,clean_description,skills_list,skills
0,Principal Data Analyst,Our mission at HubSpot is to help millions of ...,HubSpot,hubspot,https://cdn-images.himalayas.app/pgeb5qfnlbpwv...,Full Time,152200.00,243500.00,annual,[Senior],...,"[Principal-Data-Analyst, Measurement-Analyst, ...",[],"<p>Our mission at <a href=""https://himalayas.a...",1794475185,1799659185,https://himalayas.app/companies/hubspot/jobs/p...,https://himalayas.app/companies/hubspot/jobs/p...,Our mission at HubSpot is to help millions of ...,"[Go, LLM / GenAI]","Go, LLM / GenAI"
1,Competitive Data Scientist (Kaggle) – Remote (...,"IMPORTANT: after you apply, please check your ...",RemoteJobsOne,remotejobsone,https://cdn-images.himalayas.app/7fzya4qky644j...,Contractor,60.00,120.00,hourly,[Mid-level],...,"[Data-Scientist, Machine-Learning-Engineer, Qu...",[Data Science],"<p><strong>IMPORTANT: after you apply, please ...",1794472799,1799656799,https://himalayas.app/companies/remotejobsone/...,https://himalayas.app/companies/remotejobsone/...,"IMPORTANT: after you apply, please check your ...",[Python],Python
2,"Senior Scientist, Computational Pathology and ...",Career CategoryResearchJob DescriptionAt Amgen...,Amgen,amgen,https://cdn-images.himalayas.app/ygg1esm3yk1c2...,Full Time,133673.55,180852.45,annual,[Senior],...,"[Senior-Scientist, Computational-Pathologist, ...","[Research & Development, Data Science]",<h3><b>Career Category</b></h3>Research<h3><b>...,1794466407,1799650406,https://himalayas.app/companies/amgen/jobs/sen...,https://himalayas.app/companies/amgen/jobs/sen...,Career Category Research Job Description At Am...,"[Python, R, Scikit-Learn, NumPy, Computer Visi...","Python, R, Scikit-Learn, NumPy, Computer Visio..."
3,Data Scientist | Modeling,Machinify is a leading healthcare intelligence...,Machinify,machinify,https://cdn-images.himalayas.app/b9v484aojf6w5...,Full Time,180000.00,230000.00,annual,[Senior],...,"[Data-Scientist, Machine-Learning-Scientist, H...",[Data Science],"<div class=""content-intro""><p><a href=""https:/...",1794465580,1799649579,https://himalayas.app/companies/machinify/jobs...,https://himalayas.app/companies/machinify/jobs...,Machinify is a leading healthcare intelligence...,"[Python, SQL, R]","Python, SQL, R"
4,"Senior Data Scientist, Applied AI and Agentic ...",Model complex business problems and discoverin...,VSP Vision,vsp-vision,https://cdn-images.himalayas.app/qdbcj6l66q45f...,Full Time,84000.00,141750.00,annual,[Senior],...,"[Data-Scientist, Applied-AI-Engineer, AI-Solut...",[Data Science],Model complex business problems and discoverin...,1794455665,1799639664,https://himalayas.app/companies/vsp-vision/job...,https://himalayas.app/companies/vsp-vision/job...,Model complex business problems and discoverin...,"[Python, JavaScript, TypeScript, LLM / GenAI, ...","Python, JavaScript, TypeScript, LLM / GenAI, S..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7957,Lead Biostatistician (IVD Analytical),This is an exciting opportunity to advance Nat...,Natera,natera,https://cdn-images.himalayas.app/1e5nqzbnkuslv...,Full Time,130900.00,163600.00,annual,[Senior],...,"[Lead-Biostatistician, Biostatistician, Statis...",[Research & Development],<p>This is an exciting opportunity to advance ...,1794459066,1799643065,https://himalayas.app/companies/natera/jobs/le...,https://himalayas.app/companies/natera/jobs/le...,This is an exciting opportunity to advance Nat...,[R],R
7959,Principal Biology Specialist,About TaskUs:TaskUs is a provider of outsource...,TaskUs,taskus,https://cdn-images.himalayas.app/jtyt8ntn3ts50...,Full Time,150000.00,220000.00,annual,[Senior],...,"[Biology-Specialist, Biosecurity-Analyst, Biol...",[],"<h3>About <a href=""https://himalayas.app/compa...",1793256956,1798440955,https://himala

In [46]:
df_with_salary_relevant = df_with_salary[df_with_salary['skills'] != 'None Specified']

In [47]:
df_with_salary_relevant.reset_index(drop=True)

,title,excerpt,companyName,companySlug,companyLogo,employmentType,minSalary,maxSalary,salaryPeriod,seniority,...,categories,parentCategories,description,pubDate,expiryDate,applicationLink,guid,clean_description,skills_list,skills
0,Principal Data Analyst,Our mission at HubSpot is to help millions of ...,HubSpot,hubspot,https://cdn-images.himalayas.app/pgeb5qfnlbpwv...,Full Time,152200.00,243500.00,annual,[Senior],...,"[Principal-Data-Analyst, Measurement-Analyst, ...",[],"<p>Our mission at <a href=""https://himalayas.a...",1794475185,1799659185,https://himalayas.app/companies/hubspot/jobs/p...,https://himalayas.app/companies/hubspot/jobs/p...,Our mission at HubSpot is to help millions of ...,"[Go, LLM / GenAI]","Go, LLM / GenAI"
1,Competitive Data Scientist (Kaggle) – Remote (...,"IMPORTANT: after you apply, please check your ...",RemoteJobsOne,remotejobsone,https://cdn-images.himalayas.app/7fzya4qky644j...,Contractor,60.00,120.00,hourly,[Mid-level],...,"[Data-Scientist, Machine-Learning-Engineer, Qu...",[Data Science],"<p><strong>IMPORTANT: after you apply, please ...",1794472799,1799656799,https://himalayas.app/companies/remotejobsone/...,https://himalayas.app/companies/remotejobsone/...,"IMPORTANT: after you apply, please check your ...",[Python],Python
2,"Senior Scientist, Computational Pathology and ...",Career CategoryResearchJob DescriptionAt Amgen...,Amgen,amgen,https://cdn-images.himalayas.app/ygg1esm3yk1c2...,Full Time,133673.55,180852.45,annual,[Senior],...,"[Senior-Scientist, Computational-Pathologist, ...","[Research & Development, Data Science]",<h3><b>Career Category</b></h3>Research<h3><b>...,1794466407,1799650406,https://himalayas.app/companies/amgen/jobs/sen...,https://himalayas.app/companies/amgen/jobs/sen...,Career Category Research Job Description At Am...,"[Python, R, Scikit-Learn, NumPy, Computer Visi...","Python, R, Scikit-Learn, NumPy, Computer Visio..."
3,Data Scientist | Modeling,Machinify is a leading healthcare intelligence...,Machinify,machinify,https://cdn-images.himalayas.app/b9v484aojf6w5...,Full Time,180000.00,230000.00,annual,[Senior],...,"[Data-Scientist, Machine-Learning-Scientist, H...",[Data Science],"<div class=""content-intro""><p><a href=""https:/...",1794465580,1799649579,https://himalayas.app/companies/machinify/jobs...,https://himalayas.app/companies/machinify/jobs...,Machinify is a leading healthcare intelligence...,"[Python, SQL, R]","Python, SQL, R"
4,"Senior Data Scientist, Applied AI and Agentic ...",Model complex business problems and discoverin...,VSP Vision,vsp-vision,https://cdn-images.himalayas.app/qdbcj6l66q45f...,Full Time,84000.00,141750.00,annual,[Senior],...,"[Data-Scientist, Applied-AI-Engineer, AI-Solut...",[Data Science],Model complex business problems and discoverin...,1794455665,1799639664,https://himalayas.app/companies/vsp-vision/job...,https://himalayas.app/companies/vsp-vision/job...,Model complex business problems and discoverin...,"[Python, JavaScript, TypeScript, LLM / GenAI, ...","Python, JavaScript, TypeScript, LLM / GenAI, S..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3547,AI Software Engineer,Job DescriptionNetImpact Strategies is seeking...,NetImpact Strategies,netimpact-strategies,https://cdn-images.himalayas.app/8erpushw8io5e...,Full Time,107000.00,140000.00,annual,[Mid-level],...,"[Fullstack-Development, Software-Engineer, Bac...","[Developer, Data Science]","<h3>Job Description</h3><p><a href=""https://hi...",1792611123,1795140283,https://himalayas.app/companies/netimpact-stra...,https://himalayas.app/companies/netimpact-stra...,Job Description NetImpact Strategies is seekin...,"[SQL, C#, Java, AWS, Azure, Docker, Kubernetes...","SQL, C#, Java, AWS, Azure, Docker, Kubernetes,..."
3548,AI Product Manager,Position Summary The AI Product Manager serves...,St. Jude Children's Research Hospital,st-jude-children-s-research-hospital,https://cdn-images.himalayas.app/s2x0va4w9xkte...,Full Time,94640.00,169520

In [48]:
print("Currencies:\n", df_with_salary_relevant["currency"].value_counts(dropna=False))
print("\nSalary Periods:\n", df_with_salary_relevant["salaryPeriod"].value_counts(dropna=False))

Currencies:
 currency
USD    3518
CAD      23
INR       6
EUR       3
GBP       2
Name: count, dtype: int64

Salary Periods:
 salaryPeriod
annual     3494
hourly       41
monthly      16
weekly        1
Name: count, dtype: int64


In [49]:
import numpy as np
import pandas as pd

# 1. Make an explicit, cleanly indexed copy before modifying columns
df_with_salary_relevant = df_with_salary_relevant.reset_index(drop=True)

# 2. Ensure salary columns are numeric
df_with_salary_relevant["minSalary"] = pd.to_numeric(df_with_salary_relevant["minSalary"], errors="coerce")
df_with_salary_relevant["maxSalary"] = pd.to_numeric(df_with_salary_relevant["maxSalary"], errors="coerce")

# 3. Calculate midpoint
df_with_salary_relevant["midpoint_salary"] = df_with_salary_relevant[["minSalary", "maxSalary"]].mean(axis=1)

# 4. Map salaryPeriod to annual multipliers
period_clean = df_with_salary_relevant["salaryPeriod"].fillna("unknown").str.lower().str.strip()

multipliers = {
    "annual": 1,
    "yearly": 1,
    "year": 1,
    "monthly": 12,
    "month": 12,
    "weekly": 52,
    "week": 52,
    "daily": 260,
    "day": 260,
    "hourly": 2080,
    "hour": 2080,
}

df_with_salary_relevant["multiplier"] = period_clean.map(multipliers)

# Fallback when salaryPeriod is missing
df_with_salary_relevant.loc[
    df_with_salary_relevant["multiplier"].isna() & (df_with_salary_relevant["midpoint_salary"] < 500),
    "multiplier",
] = 2080
df_with_salary_relevant.loc[
    df_with_salary_relevant["multiplier"].isna() & (df_with_salary_relevant["midpoint_salary"] >= 15000),
    "multiplier",
] = 1

# 5. Compute annualized salary
df_with_salary_relevant["annual_salary"] = (
    df_with_salary_relevant["midpoint_salary"] * df_with_salary_relevant["multiplier"])

In [50]:
df_with_salary_relevant = df_with_salary_relevant[~df_with_salary_relevant['employmentType'].isin(['Contractor', 'Part Time', 'Temporary'])].reset_index()

In [51]:
df_with_salary_relevant['employmentType'].value_counts()

,count
employmentType,
Full Time,3442
Other,37
Intern,23


In [52]:
df_with_salary_relevant.iloc[0]

,0
index,0
title,Principal Data Analyst
excerpt,Our mission at HubSpot is to help millions of ...
companyName,HubSpot
companySlug,hubspot
companyLogo,https://cdn-images.himalayas.app/pgeb5qfnlbpwv...
employmentType,Full Time
minSalary,152200.0
maxSalary,243500.0
salaryPeriod,annual


In [54]:
df_with_salary_relevant = df_with_salary_relevant.drop(columns=['excerpt', 'companyLogo', 'description', 'applicationLink', 'guid', 'multiplier'])

In [58]:
df_with_salary_relevant = df_with_salary_relevant[df_with_salary_relevant['currency'] == 'USD'].reset_index()

In [60]:
df_with_salary_relevant.to_csv('web_analytics_dataset.csv', index=False)